<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/ProteinMPNN%20%26%20AlphaFold%20Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 1. One-Click Environment Setup (~3-5 min)
#@markdown This cell initializes Google Drive, downloads AlphaFold weights, and configures ProteinMPNN/ColabDesign dependencies.

import os
import sys
import time
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
  print("Mounting Google Drive...")
  drive.mount("/content/drive")

# 2. Install ColabDesign
if not os.path.isdir("colabdesign"):
  print("Installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")
if "colabdesign" not in sys.path:
  sys.path.append("colabdesign")

# 3. Clone ProteinMPNN repository
if not os.path.isdir("ProteinMPNN"):
  print("Cloning ProteinMPNN...")
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")

# 4. Download ProteinMPNN weights
os.makedirs("colabdesign/params", exist_ok=True)
if not os.path.exists("colabdesign/params/protein_mpnn_20220829.tar.gz"):
  print("Downloading ProteinMPNN weights...")
  os.system("wget -qnc -P colabdesign/params/ https://files.ipd.uw.edu/pub/protein_mpnn/protein_mpnn_20220829.tar.gz")
  os.system("tar -xzf colabdesign/params/protein_mpnn_20220829.tar.gz -C colabdesign/params/")

# 5. Download AlphaFold Weights using high-speed aria2
if not os.path.isdir("params"):
  os.makedirs("params", exist_ok=True)
if not os.path.isfile("params/done.txt"):
  print("Downloading AlphaFold parameters (approx. 3-4 GB)...")
  os.system("apt-get install -y aria2")
  os.system("aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
  os.system("tar -xf alphafold_params_2022-12-06.tar -C params")
  os.system("touch params/done.txt")
  os.system("rm alphafold_params_2022-12-06.tar")

print("\n[SETUP SUCCESS] Your environment is fully configured and ready to run design!")

Mounting Google Drive...
Mounted at /content/drive
Installing ColabDesign...
Cloning ProteinMPNN...

[SETUP SUCCESS] Your environment is fully configured and ready to run design!


In [ ]:
%%time
#@title 2. Run ProteinMPNN & AlphaFold Validation (with Real-time Auto-Save)
#@markdown Ensure you completed Step 1 (Setup) before running this cell.

#@markdown ### Design Settings
num_seqs = 8 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
mpnn_sampling_temp = 0.1 #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5", "1.0"] {type:"raw"}
rm_aa = "C" #@param {type:"string"}
use_solubleMPNN = False #@param {type:"boolean"}

#@markdown ### Binder Design Settings (Chain Selection)
binder_chain = "A" #@param {type:"string"}
#@markdown - The scaffold/binder chain you want to redesign
target_chain = "B" #@param {type:"string"}
#@markdown - The reference target protein chain to keep fixed (both sequence & structure)

#@markdown ### Validation Settings
initial_guess = False #@param {type:"boolean"}
num_recycles = 1 #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
use_multimer = False #@param {type:"boolean"}

#@markdown ### Google Drive Directory Paths
pdb_drive_path = "/content/drive/MyDrive/your_pdbs_folder_or_file.pdb" #@param {type:"string"}
#@markdown - Destination path to auto-save and stream all design & validation scores dynamically
save_drive_dir = "/content/drive/MyDrive/MPNN" #@param {type:"string"}

import os
import glob
import shutil
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.protein import parse_pdb

os.makedirs(save_drive_dir, exist_ok=True)

# 1. Determine files to process
if not os.path.exists(pdb_drive_path):
  raise FileNotFoundError(f"Input path does not exist: {pdb_drive_path}")

pdb_files = []
if os.path.isdir(pdb_drive_path):
  pdb_files = sorted(glob.glob(os.path.join(pdb_drive_path, "*.pdb")))
  print(f"Batch mode: Found {len(pdb_files)} PDBs.")
else:
  pdb_files = [pdb_drive_path]
  print("Single file mode.")

copies = 1
num_designs = 1

# 2. Sequential execution & Instant Sync
for i, pdb_file in enumerate(pdb_files):
  path = os.path.splitext(os.path.basename(pdb_file))[0]
  print(f"\n>>> [Processing {i+1}/{len(pdb_files)}] Target: {path}")

  local_out_dir = f"outputs/{path}"
  drive_out_dir = os.path.join(save_drive_dir, path)
  os.makedirs(local_out_dir, exist_ok=True)
  os.makedirs(drive_out_dir, exist_ok=True)

  # Determine contigs format automatically to fix target chain and redesign binder chain
  # Typically format is: [target_chain]/[binder_length] or similar contig mapping
  # We read the PDB to get residue counts for the chains
  try:
    pdb_parsed = parse_pdb(pdb_file)
    binder_len = len(pdb_parsed["seq"][pdb_parsed["idx"]["chain"] == binder_chain])

    # Setup contig argument so target_chain is fixed and binder_chain is redesigned
    # Example syntax for ColabDesign: B/100 (where B is fixed target, and 100 residues binder is redesigned)
    contigs_str = f"{target_chain}/{binder_len}"
    print(f"Auto-configured contig for Binder design: {contigs_str} (Target Chain: {target_chain}, Binder redesigned length: {binder_len})")
  except Exception as e:
    print(f"Warning parsing chains from PDB: {e}. Falling back to default contig '100'.")
    contigs_str = "100"

  # Construct options for ColabDesign Validation Script
  opts = [
    f"--pdb={pdb_file}",
    f"--loc=outputs/{path}",
    f"--contig={contigs_str}",
    f"--copies={copies}",
    f"--num_seqs={num_seqs}",
    f"--num_recycles={num_recycles}",
    f"--rm_aa={rm_aa}",
    f"--mpnn_sampling_temp={mpnn_sampling_temp}",
    f"--num_designs={num_designs}"
  ]

  if initial_guess: opts.append("--initial_guess")
  if use_multimer: opts.append("--use_multimer")
  if use_solubleMPNN: opts.append("--use_soluble")

  opts_str = ' '.join(opts)
  print(f"Executing designability test...")
  os.system(f"python colabdesign/rf/designability_test.py {opts_str}")

  # Copy files immediately to drive to secure progress
  print(f"Securing data to Google Drive for: {path}...")
  if os.path.exists(local_out_dir):
    for file_name in os.listdir(local_out_dir):
      src_file = os.path.join(local_out_dir, file_name)
      dest_file = os.path.join(drive_out_dir, file_name)
      if os.path.isfile(src_file):
        shutil.copy2(src_file, dest_file)

    # Sync design best outputs
    for local_file in glob.glob(f"outputs/{path}_*"):
      shutil.copy2(local_file, save_drive_dir)

  print(f"✓ Processed and backed up {path} safely.")

print("\nBatch design and structural validation completed successfully!")

In [ ]:
#@title 3. 📊 Results Leaderboard Generator (Extract & Rank Best Designs)
#@markdown 运行此单元格将自动扫描您的 `save_drive_dir` 目录，并将所有 PDB 验证的分数（pLDDT, RMSD 等）提取并合并，生成一个便于筛选的高清排行榜表格。

import os
import glob
import pandas as pd
from IPython.display import display

# 建议与步骤 2 中的路径保持一致
results_folder = "/content/drive/MyDrive/MPNN"

if not os.path.exists(results_folder):
    print(f"❌ 路径不存在: {results_folder}，请先确保步骤2有运行结果并成功同步到云盘中。")
else:
    print(f"🔍 正在扫描文件夹: {results_folder} 下的所有设计报告...")

    all_data = []
    # 搜寻所有子目录下的 csv 或者是 json 报告
    report_files = glob.glob(os.path.join(results_folder, "**", "*.csv"), recursive=True) + \
                   glob.glob(os.path.join(results_folder, "**", "*.json"), recursive=True)

    if not report_files:
        print("⚠️ 暂未在您的云盘文件夹中检测到任何验证数据。请等待步骤2运行产生数据并成功同步后再试。")
    else:
        for f in report_files:
            try:
                # 根据文件类型读取数据
                if f.endswith('.csv'):
                    df_temp = pd.read_csv(f)
                else:
                    df_temp = pd.read_json(f)

                # 记录对应设计的蛋白名称
                parent_name = os.path.basename(os.path.dirname(f))
                df_temp['Target_Protein'] = parent_name
                all_data.append(df_temp)
            except Exception as e:
                continue

        if all_data:
            # 合并数据
            leaderboard = pd.concat(all_data, ignore_index=True)

            # 整理常用列顺序，方便直观对比
            preferred_cols = ['Target_Protein', 'seq', 'score', 'plddt', 'rmsd', 'seq_recovery']
            existing_cols = [col for col in preferred_cols if col in leaderboard.columns]
            other_cols = [col for col in leaderboard.columns if col not in existing_cols]
            leaderboard = leaderboard[existing_cols + other_cols]

            # 按照结构折叠置信度 pLDDT (越高越好) 以及 RMSD (越低越好) 排序
            if 'plddt' in leaderboard.columns:
                leaderboard = leaderboard.sort_values(by='plddt', ascending=False)
            elif 'score' in leaderboard.columns:
                leaderboard = leaderboard.sort_values(by='score', ascending=True)

            print(f"\n🎉 成功汇总 {len(leaderboard)} 条设计序列结果！以下为您按结构预测置信度排序后的前 15 个设计：\n")

            # 针对 Colab 进行美化展示
            pd.set_option('display.max_colwidth', None)
            display(leaderboard.head(15).style.background_gradient(cmap='viridis', subset=['plddt'] if 'plddt' in leaderboard.columns else []))
        else:
            print("⚠️ 未能成功解析任何有效的数据记录。")